In [ ]:
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec


# Granite Speech Audio Playground

## Setup

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, device_map=device,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
).eval()

## Test Data Setup

In [ ]:
import torchaudio
from IPython.display import Audio, display


dataset = []

dataset_path = "/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test" #other languages *_de *_fr *_es
# other datasets
# Mozilla Commonvoice: /training-1/asr/kaldi_data/cv19/en/test /de/ /fr/ /es/
# Mozilla Commonvoice Single Words: /training-1/asr/kaldi_data/singleword/en/test /de/ /fr/ /es/ 
# Multilingual Librispeech: /training-1/asr/kaldi_data/mls/en/test /de/ /fr/ /es/
# Voxpopuli: /training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test *_de *_fr *_es

wav_scp_dict = {}
with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
    for line in rf:
        id, wav_path = line.strip().split("\t")
        wav_scp_dict[id] = wav_path

text_dict = {}
with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
    for line in rf:
        id, text = line.strip().split("\t")
        text_dict[id] = text

for id, wav_path in wav_scp_dict.items():
    text = text_dict[id]
    dataset.append({"audio" : wav_path, "text" : text})

#helper function: load audio and make sure it is at 16kHz
def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform
    
first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = load_audio(first_entry["audio"])
display(Audio(audio, rate=16000))



## Transcription Function (Prompt Changeable)

In [4]:
def transcribe(wav, prompt="can you transcribe the speech into a written format?", hotwords=None):
    if hotwords:
        prompt = f"{prompt}\nhotwords: {hotwords}"
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{prompt}"},
    ]
    text = processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = processor(text, wav, device=device, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1)
    return processor.tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:],
                                      skip_special_tokens=True).strip()

## Test on a Single Audio from the Test Dataset

In [ ]:
import torchaudio

sample = dataset[0]

wav = load_audio(sample["audio"])
model_output = transcribe(wav, prompt="can you transcribe the speech into a written format?")

print("SAMPLE :", sample)
print("LABEL :", sample["text"])
print("MODEL_OUTPUT :", model_output)

## Adding Hotwords

In [ ]:

sample_path = "/training-1/asr/voxpopuli/out/transcribed_data/en/2018/20180704-0900-PLENARY-en_20180704-17:51:01_12.wav"
sample_text = "without these mistakes that you made we would not be debating this issue here today so what im asking mr kurz to do is to upscale this strategy to an eur one hundred and twenty million anti poverty strategy for all european citizens"

wav = load_audio(sample_path)
display(Audio(wav, rate=16000))

prompt = "can you transcribe the speech into a written format?"
hotwords = "mr kurz"

print(transcribe(wav, prompt=prompt, hotwords=None))
print("-"*10)
print(transcribe(wav, prompt=prompt, hotwords=hotwords))

## Compare Prompt Variants

In [ ]:
prompts = [
    "can you transcribe the speech into a written format?",
    "Transcribe verbatim, including hesitations and filler words.",
    "Transcribe with proper punctuation and capitalization.",
]

for p in prompts:
    print("=" * 70)
    print("PROMPT:", p)
    print(transcribe(wav, prompt=p, hotwords=hotwords))